In [ ]:
from pathlib import Path
import json, os, re
from typing import Any, Dict, Optional
from openai import OpenAI

# 输入 / 输出路径
INPUT_JSON = Path("")
OUTPUT_JSON = INPUT_JSON.parent / "judgement.json"  # 与输入同文件夹

# JSON 字段名
FIELD_TITLE = "title"
FIELD_BODY  = "body"
FIELD_TRACE = "gen_trace"  # 读取 trace 的字段

# OpenAI
MODEL_NAME = "gpt-5"
api_key = ""  # 请替换为你自己的 API key
client = OpenAI(api_key=api_key)


In [2]:
# %% 重新定义（修复）模板 —— 只需运行这个 cell 覆盖原变量
# %% 重新定义 Step1 Prompt（召回优先 & 保持 is_bug + bugs）
STEP1_PROMPT_TEMPLATE = (
    "You are a QA triager. Be RECALL-ORIENTED: if the trace suggests any plausible non-crash functional failure, "
    "classify as bug and list it/them. Step2 will verify against the Bug Report.\n"
    "Treat as NCF (examples include but not limited to): tap/click triggers no expected change; action reports success "
    "but state/output unchanged; toggles/spinners/selections not applied; navigation opens wrong/blank destination; "
    "save/export/share produces missing/empty/wrong file; input accepted but not reflected.\n"
    "DO NOT count: repeated/long-lived toast artifacts; pure UX-only issues (inconvenient but functionally correct).\n"
    "If uncertain between UX vs NCF, TREAT AS NCF here.\n"
    "Output ONLY a JSON object with EXACTLY these two keys:\n"
    '{{ "is_bug": "yes" | "no", "bugs": [ {{ "title": "short name", "description": "concise functional failure" }} ] }}\n'
    'If "is_bug" = "no", then "bugs" MUST be an empty list. Order bugs from most severe/central to least.\n'
    "Trace:\n{trace}\n"
)

STEP2_PROMPT_TEMPLATE = (
    "Given a Bug Report (title + body) and a set of detected NCF bugs, pick the SINGLE bug that best aligns with the BR.\n"
    "Focus on functional correctness; ignore pure UX-only concerns.\n"
    "Return ONLY a JSON object with:\n"
    "{{\n"
    '  "same_as_BR": "yes" | "no" | "partial",\n'
    '  "matched_bug_index": integer or null,\n'
    '  "matched_bug": {{ "title": string, "description": string }} or null,\n'
    '  "reason": "brief why this choice aligns (or not)"\n'
    "}}\n"
    "BR Title:\n{br_title}\n\n"
    "BR Body:\n{br_body}\n\n"
    "Detected bugs (JSON):\n{bugs_json}\n"
)


In [3]:
def safe_json_loads(s: str) -> Optional[dict]:
    try:
        return json.loads(s)
    except Exception:
        pass
    fence = re.compile(r"^\s*```(?:json)?\s*([\s\S]*?)\s*```\s*$", re.IGNORECASE)
    m = fence.match(s or "")
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            return None
    return None

def normalize_yes_no(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"yes","y","true","1"}: return True
        if v in {"no","n","false","0"}: return False
    return False

# %% 精简但带“reasoning summary”的 Step1 / Step2 封装
def call_step1(trace_text: str):
    """
    返回三件事：
    - parsed: 解析后的 dict，只保留 {"is_bug": "yes|no", "bugs": [{title,description}, ...]}
    - raw_text: 模型原始 output_text
    - reasoning_summaries: 模型 reasoning.summary 列表
    """
    prompt = STEP1_PROMPT_TEMPLATE.format(trace=(trace_text or "").strip())
    resp = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        text={"format": {"type": "json_object"}},  # JSON 模式
        instructions="Return a single JSON object with exactly 'is_bug' and 'bugs'. No extra text.",
        reasoning={"summary": "auto"},
    )

    # 收集 reasoning summary
    reasoning_summaries = []
    for out in getattr(resp, "output", []):
        if getattr(out, "type", None) == "reasoning":
            for s in getattr(out, "summary", []) or []:
                txt = getattr(s, "text", None)
                if txt:
                    reasoning_summaries.append(txt)

    raw_text = getattr(resp, "output_text", "") or ""
    data = safe_json_loads(raw_text)
    if not isinstance(data, dict):
        parsed = {"is_bug": "no", "bugs": []}
        return parsed, raw_text, reasoning_summaries

    # 只保留 schema，并做最小清洗
    is_bug = str(data.get("is_bug", "no")).strip().lower()
    bugs = data.get("bugs", [])
    if not isinstance(bugs, list):
        bugs = []
    cleaned = [
        {
            "title": str(b.get("title", "")).strip(),
            "description": str(b.get("description", "")).strip()
        }
        for b in bugs if isinstance(b, dict)
    ]
    # 一致性：is_bug=yes 但无候选 → 视为 no
    if is_bug in {"yes","true","1"} and not cleaned:
        is_bug = "no"

    parsed = {"is_bug": "yes" if is_bug in {"yes","true","1"} else "no", "bugs": cleaned}
    return parsed, raw_text, reasoning_summaries


def call_step2_pick_match(br_title: str, br_body: str, bugs_payload: Dict[str, Any]):
    """
    返回三件事：
    - parsed: {"same_as_BR": "yes|no|partial", "matched_bug_index": int|None, "matched_bug": {...}|None, "reason": str}
    - raw_text: 模型原始 output_text
    - reasoning_summaries: 模型 reasoning.summary 列表
    """
    bugs_json = json.dumps(bugs_payload, ensure_ascii=False, indent=2)
    prompt = STEP2_PROMPT_TEMPLATE.format(
        br_title=br_title or "",
        br_body=br_body or "",
        bugs_json=bugs_json
    )
    resp = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        text={"format": {"type": "json_object"}},
        instructions="Return a single JSON object with keys: same_as_BR, matched_bug_index, matched_bug, reason.",
        reasoning={"summary": "auto"},
    )

    reasoning_summaries = []
    for out in getattr(resp, "output", []):
        if getattr(out, "type", None) == "reasoning":
            for s in getattr(out, "summary", []) or []:
                txt = getattr(s, "text", None)
                if txt:
                    reasoning_summaries.append(txt)

    raw_text = getattr(resp, "output_text", "") or ""
    data = safe_json_loads(raw_text)
    if not isinstance(data, dict):
        parsed = {"same_as_BR": "no", "matched_bug_index": None, "matched_bug": None, "reason": "model output invalid"}
        return parsed, raw_text, reasoning_summaries

    mb = data.get("matched_bug")
    if isinstance(mb, dict):
        mb = {
            "title": (mb.get("title") or "").strip(),
            "description": (mb.get("description") or "").strip(),
        }
    else:
        mb = None

    parsed = {
        "same_as_BR": data.get("same_as_BR", "no"),
        "matched_bug_index": data.get("matched_bug_index", None),
        "matched_bug": mb,
        "reason": data.get("reason", "")
    }
    return parsed, raw_text, reasoning_summaries


In [4]:
# 读取
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    items = json.load(f)

if not isinstance(items, list):
    raise ValueError("输入 JSON 顶层必须是 list。")

total = len(items)

for idx, item in enumerate(items, start=1):
    title = item.get(FIELD_TITLE, "") or ""
    body  = item.get(FIELD_BODY, "") or ""
    trace = item.get(FIELD_TRACE, "") or ""

    # ✅ 正确的进度打印
    print(f"Proceeding {idx}/{total} {title[:60]}")

    # Step1
    if not trace.strip():
        step1_parsed, step1_raw, step1_summaries = {"is_bug": "no", "bugs": []}, "", []
    else:
        step1_parsed, step1_raw, step1_summaries = call_step1(trace)

    is_bug = normalize_yes_no(step1_parsed.get("is_bug"))

    # Step2（仅在有 bug 时）
    if is_bug and step1_parsed.get("bugs"):
        step2_parsed, step2_raw, step2_summaries = call_step2_pick_match(title, body, step1_parsed)
        same_as_BR = step2_parsed.get("same_as_BR", "")
        matched = step2_parsed.get("matched_bug")
        if matched and matched.get("description"):
            reason_text = matched["description"]
        else:
            first = step1_parsed["bugs"][0] if step1_parsed["bugs"] else {}
            reason_text = (first.get("description") or "").strip()
    else:
        step2_parsed, step2_raw, step2_summaries = None, "", []
        same_as_BR = ""
        reason_text = "no NCF bug detected"

    # 写入 judgement
    item["judgement"] = {
        "is_ncf_bug": "yes" if is_bug else "no",
        "same_as_BR": same_as_BR,
        "reason": reason_text
    }

    # 保存每一步的 LLM 输出和 reasoning
    item["step1_llm_output"] = step1_raw
    item["step1_reasoning_summary"] = step1_summaries
    if step2_parsed is not None:
        item["step2_llm_output"] = step2_raw
        item["step2_reasoning_summary"] = step2_summaries

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(items, f, ensure_ascii=False, indent=2)

print(f"完成：已写入 {OUTPUT_JSON}")


Proceeding 1/491 Error in One Site
Proceeding 2/491 [BUG] When you import 2 filter lists, the second list replac
Proceeding 3/491 Overchan does not display browsing history 
Proceeding 4/491 Can't create new files in overchan
Proceeding 5/491 Study Pad: heading (label) text does not update when the lab
Proceeding 6/491 Some Hebrew words don't open dictionary in strongs underline
Proceeding 7/491 Strongs numbers can be changed even if strongs are not avail
Proceeding 8/491 Skipping last episode in queue leaves episode in queue, unpl
Proceeding 9/491 Speed button doesn't respond immediately after Sonic (de)act
Proceeding 10/491 Location streaming is missing from ArcaneChat v1.52.1+
Proceeding 11/491 [BUG] En la sección de copias y seguridad
Proceeding 12/491 [BUG] Only Sporadic to play playlist without shuffle/random 
Proceeding 13/491 Adding photo does not work when just creating a new card
Proceeding 14/491 No numbers displayed in recent calls list
Proceeding 15/491 Delete contact bloc